# TCN Multi-Output Forecasting — All 21 Patients

**Strategy**: Multi-Output (MIMO) only  
**Horizons**: 1, 10, 15, 20 steps  
**Patients**: All 21 MIT-BIH patients (excluding 111 & 118)  
**Epochs**: 30 (uniform across all runs)  

Metrics: RMSE · MAE · R² · Training Time

In [ ]:
# ── Install & Imports ────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'wfdb', 'tqdm', 'seaborn', '--quiet'])

import os, time, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import wfdb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)

SEED = 42
np.random.seed(SEED)
print('All imports OK.')

In [ ]:
# ── TensorFlow Setup ─────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv1D, Dense, Dropout, Add, Activation, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU(s): {[g.name for g in gpus]}')
else:
    print('No GPU — training will be slow.')

## 1. Configuration

In [ ]:
# ── Dataset path ─────────────────────────────────────────────────────────────
DATA_DIR = r"/kaggle/input/datasets/rracer17/mit-bih-mitdb/mit-bih-arrhythmia-database-1.0.0"

# ── Paper constants (Section 3, Dudukcu et al. 2023) ────────────────────────
FS           = 360
TOTAL_STEPS  = 100_000
TRAIN_STEPS  = 40_000
VAL_STEPS    = 10_000
TEST_STEPS   = 50_000
LOOKBACK     = 10

# ── All 21 patients (excluding 111 & 118) ───────────────────────────────────
PATIENTS = [
    '100', '101', '102', '103', '104', '105',
    '106', '107', '108', '109', '112', '113',
    '114', '115', '116', '117', '119',
    '121', '122', '123', '124'
]
assert len(PATIENTS) == 21

# ── Forecasting config ───────────────────────────────────────────────────────
HORIZONS      = [1, 10, 15, 20]
EPOCHS        = 30
BATCH_SIZE    = 256
NUM_FILTERS   = 64
KERNEL_SIZE   = 3
NUM_BLOCKS    = 3       # dilations: 1, 2, 4
DROPOUT_RATE  = 0.1
LEARNING_RATE = 1e-3

print(f'Patients  : {len(PATIENTS)}')
print(f'Horizons  : {HORIZONS}')
print(f'Strategy  : Multi-Output only')
print(f'TCN       : filters={NUM_FILTERS}, kernel={KERNEL_SIZE}, blocks={NUM_BLOCKS}')
print(f'Training  : epochs={EPOCHS}, batch={BATCH_SIZE}, lr={LEARNING_RATE}')

## 2. Data Loading & Preprocessing

In [ ]:
def load_ecg_signal(record_id, data_dir, n_steps=100_000):
    """Load MLII lead from MIT-BIH, truncate/pad to n_steps."""
    path = os.path.join(data_dir, record_id)
    rec  = wfdb.rdrecord(path)
    sig_names_upper = [s.upper() for s in rec.sig_name]
    ch = sig_names_upper.index('MLII') if 'MLII' in sig_names_upper else 0
    signal = rec.p_signal[:, ch].astype(np.float32)
    if len(signal) < n_steps:
        pad = np.full(n_steps - len(signal), signal[-1], dtype=np.float32)
        signal = np.concatenate([signal, pad])
    return signal[:n_steps]


def preprocess_patient(signal, train_steps=40_000, val_steps=10_000):
    """Split → train/val/test, MinMax-normalise (fit on train only)."""
    train_end = train_steps
    val_end   = train_steps + val_steps
    train_raw = signal[:train_end]
    val_raw   = signal[train_end:val_end]
    test_raw  = signal[val_end:]
    scaler     = MinMaxScaler(feature_range=(0, 1))
    train_norm = scaler.fit_transform(train_raw.reshape(-1, 1)).flatten()
    val_norm   = scaler.transform(val_raw.reshape(-1, 1)).flatten()
    test_norm  = scaler.transform(test_raw.reshape(-1, 1)).flatten()
    return train_norm, val_norm, test_norm, scaler


def make_multistep_sequences(signal, lookback, horizon):
    """X = lookback window, y = next H steps (Multi-Output)."""
    X, y = [], []
    for i in range(len(signal) - lookback - horizon + 1):
        X.append(signal[i : i + lookback])
        y.append(signal[i + lookback : i + lookback + horizon])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


print('Preprocessing functions defined.')

In [ ]:
# ── Load all 21 patients ─────────────────────────────────────────────────────
patient_signals = {}

print(f'Loading {len(PATIENTS)} patients...\n')
for rid in tqdm(PATIENTS, desc='Patients'):
    signal = load_ecg_signal(rid, DATA_DIR, n_steps=TOTAL_STEPS)
    tr, vl, te, sc = preprocess_patient(signal, TRAIN_STEPS, VAL_STEPS)
    patient_signals[rid] = {'train': tr, 'val': vl, 'test': te, 'scaler': sc}
    tqdm.write(f'  Patient {rid:>3s} | train={len(tr):,}  val={len(vl):,}  test={len(te):,}')

print(f'\nAll {len(patient_signals)} patients loaded.')

## 3. TCN Architecture

In [ ]:
def residual_block(x, filters, kernel_size, dilation_rate, dropout_rate):
    out = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='causal')(x)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dropout(dropout_rate)(out)
    out = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='causal')(out)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dropout(dropout_rate)(out)
    if x.shape[-1] != filters:
        x = Conv1D(filters, 1)(x)
    return Add()([x, out])


def build_tcn(lookback, output_size):
    inp = Input(shape=(lookback, 1))
    x = inp
    for i in range(NUM_BLOCKS):
        x = residual_block(x, NUM_FILTERS, KERNEL_SIZE, 2 ** i, DROPOUT_RATE)
    x = x[:, -1, :]  # last time-step
    x = Dense(NUM_FILTERS, activation='relu')(x)
    out = Dense(output_size)(x)
    model = Model(inp, out, name=f'TCN_out{output_size}')
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE), loss='mse')
    return model


def train_tcn(model, X_train, y_train, X_val, y_val, epochs=EPOCHS, patience=5):
    X_tr = X_train.reshape(-1, X_train.shape[1], 1)
    X_vl = X_val.reshape(-1, X_val.shape[1], 1)
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=max(patience // 2, 2), min_lr=1e-6),
    ]
    history = model.fit(
        X_tr, y_train, validation_data=(X_vl, y_val),
        epochs=epochs, batch_size=BATCH_SIZE,
        callbacks=callbacks, verbose=0
    )
    return history


# Show architecture for H=1
_sample = build_tcn(LOOKBACK, 1)
_sample.summary()
del _sample
tf.keras.backend.clear_session()

## 4. Metrics

In [ ]:
def compute_metrics(y_true, y_pred):
    yt, yp = y_true.flatten(), y_pred.flatten()
    return {
        'RMSE': float(np.sqrt(mean_squared_error(yt, yp))),
        'MAE':  float(mean_absolute_error(yt, yp)),
        'R2':   float(r2_score(yt, yp)),
    }


def compute_per_step_metrics(y_true, y_pred):
    rows = []
    for h in range(y_true.shape[1]):
        m = compute_metrics(y_true[:, h], y_pred[:, h])
        m['Step'] = h + 1
        rows.append(m)
    return pd.DataFrame(rows)


print('Metrics functions defined.')

## 5. Training Loop — Multi-Output, All 21 Patients × 4 Horizons

In [ ]:
results          = []
per_step_results = {}
saved_preds      = {}
training_histories = {}

os.makedirs('plots', exist_ok=True)

total_runs = len(PATIENTS) * len(HORIZONS)
print(f'Running: {len(PATIENTS)} patients × {len(HORIZONS)} horizons = {total_runs} models')
print('=' * 80)

for rid in tqdm(PATIENTS, desc='Patients'):
    tr = patient_signals[rid]['train']
    vl = patient_signals[rid]['val']
    te = patient_signals[rid]['test']

    for horizon in HORIZONS:
        # Build sequences
        X_tr, y_tr = make_multistep_sequences(tr, LOOKBACK, horizon)
        X_vl, y_vl = make_multistep_sequences(vl, LOOKBACK, horizon)
        X_te, y_te = make_multistep_sequences(te, LOOKBACK, horizon)

        # Train
        t0 = time.time()
        model = build_tcn(LOOKBACK, horizon)
        history = train_tcn(model, X_tr, y_tr, X_vl, y_vl)
        preds = model.predict(
            X_te.reshape(-1, X_te.shape[1], 1),
            batch_size=2048, verbose=0
        )
        elapsed = time.time() - t0

        # Metrics
        m = compute_metrics(y_te, preds)
        m['Time_s'] = round(elapsed, 2)
        results.append({'Patient': rid, 'Horizon': horizon, **m})

        # Per-step metrics (skip for H=1)
        if horizon > 1:
            per_step_results[(rid, horizon)] = compute_per_step_metrics(y_te, preds)

        # Save predictions & history for viz
        saved_preds[(rid, horizon)] = (y_te.copy(), preds.copy())
        training_histories[(rid, horizon)] = {
            'loss': history.history['loss'],
            'val_loss': history.history['val_loss'],
        }

        del model
        gc.collect()

        tqdm.write(
            f'  Patient {rid} H={horizon:2d} | '
            f'R²={m["R2"]:.4f}  RMSE={m["RMSE"]:.4f}  MAE={m["MAE"]:.4f}  '
            f'({elapsed:.1f}s)'
        )

    tf.keras.backend.clear_session()
    gc.collect()

print(f'\nAll {len(results)} experiments complete.')

## 6. Results Summary

In [ ]:
df_results = pd.DataFrame(results)
df_results.to_csv('tcn_multioutput_results.csv', index=False)
print('Results saved → tcn_multioutput_results.csv\n')

# Average across patients per horizon
df_avg = (
    df_results
    .groupby('Horizon')
    .agg(
        RMSE_mean=('RMSE', 'mean'), RMSE_std=('RMSE', 'std'),
        MAE_mean=('MAE', 'mean'),   MAE_std=('MAE', 'std'),
        R2_mean=('R2', 'mean'),     R2_std=('R2', 'std'),
        Time_mean=('Time_s', 'mean'),
    )
    .round(4)
)
print('Average Results Across All 21 Patients (Multi-Output)')
print('=' * 70)
display(df_avg)

# Per-horizon pivot
for h in HORIZONS:
    print(f'\n{"─"*30} Horizon {h} {"─"*30}')
    df_h = df_results[df_results['Horizon'] == h]
    display(df_h[['Patient', 'RMSE', 'MAE', 'R2', 'Time_s']].set_index('Patient').round(4))

---

## 7. Visualizations

### 7.1 — R² Heatmap: Patients × Horizons
Quick overview of which patient–horizon combos perform well or poorly.

In [ ]:
# ── 7.1  R² Heatmap ──────────────────────────────────────────────────────────
pivot_r2 = df_results.pivot(index='Patient', columns='Horizon', values='R2')

fig, ax = plt.subplots(figsize=(8, 12))
sns.heatmap(
    pivot_r2, annot=True, fmt='.3f', cmap='RdYlGn', center=0.5,
    linewidths=0.5, ax=ax, vmin=0, vmax=1,
    cbar_kws={'label': 'R² Score'}
)
ax.set_title('R² Score — All Patients × Horizons\n(Multi-Output TCN)', fontsize=14, fontweight='bold')
ax.set_ylabel('Patient ID')
ax.set_xlabel('Forecast Horizon (steps)')
plt.tight_layout()
plt.savefig('plots/heatmap_r2.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.2 — RMSE Heatmap

In [ ]:
# ── 7.2  RMSE Heatmap ────────────────────────────────────────────────────────
pivot_rmse = df_results.pivot(index='Patient', columns='Horizon', values='RMSE')

fig, ax = plt.subplots(figsize=(8, 12))
sns.heatmap(
    pivot_rmse, annot=True, fmt='.4f', cmap='YlOrRd',
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'RMSE'}
)
ax.set_title('RMSE — All Patients × Horizons\n(Multi-Output TCN)', fontsize=14, fontweight='bold')
ax.set_ylabel('Patient ID')
ax.set_xlabel('Forecast Horizon (steps)')
plt.tight_layout()
plt.savefig('plots/heatmap_rmse.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.3 — Box Plots: Metric Distributions Across Patients per Horizon

In [ ]:
# ── 7.3  Box plots ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, metric in zip(axes, ['RMSE', 'MAE', 'R2']):
    sns.boxplot(
        data=df_results, x='Horizon', y=metric, ax=ax,
        palette='Set2', width=0.5
    )
    sns.stripplot(
        data=df_results, x='Horizon', y=metric, ax=ax,
        color='black', alpha=0.4, size=4, jitter=True
    )
    ax.set_title(f'{metric} Distribution by Horizon', fontsize=12, fontweight='bold')
    ax.set_xlabel('Horizon (steps)')
    ax.set_ylabel(metric)

fig.suptitle('Multi-Output TCN — Metric Distributions (21 Patients)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/boxplots_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.4 — Bar Chart: Mean Metrics by Horizon (with error bars)

In [ ]:
# ── 7.4  Bar chart with error bars ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric in zip(axes, ['RMSE', 'MAE', 'R2']):
    means = df_results.groupby('Horizon')[metric].mean()
    stds  = df_results.groupby('Horizon')[metric].std()
    colors = sns.color_palette('viridis', len(HORIZONS))

    bars = ax.bar(range(len(HORIZONS)), means.values, yerr=stds.values,
                  capsize=5, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(len(HORIZONS)))
    ax.set_xticklabels([str(h) for h in HORIZONS])
    ax.set_title(f'Mean {metric} by Horizon', fontsize=12, fontweight='bold')
    ax.set_xlabel('Horizon (steps)')
    ax.set_ylabel(metric)

    # Annotate bars with values
    for bar, val in zip(bars, means.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)

fig.suptitle('Multi-Output TCN — Mean Metrics Across 21 Patients',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/barchart_mean_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.5 — Per-Step RMSE Degradation Curves
Shows how prediction error grows with each additional step into the future.

In [ ]:
# ── 7.5  Per-step RMSE degradation ────────────────────────────────────────────
multi_step_horizons = [h for h in HORIZONS if h > 1]
fig, axes = plt.subplots(1, len(multi_step_horizons),
                         figsize=(7 * len(multi_step_horizons), 5), sharey=True)

if len(multi_step_horizons) == 1:
    axes = [axes]

for idx, horizon in enumerate(multi_step_horizons):
    ax = axes[idx]
    step_dfs = [per_step_results[(rid, horizon)]
                for rid in PATIENTS
                if (rid, horizon) in per_step_results]
    if not step_dfs:
        continue
    combined = pd.concat(step_dfs)

    # Mean line
    avg_by_step = combined.groupby('Step')['RMSE'].mean()
    std_by_step = combined.groupby('Step')['RMSE'].std()

    ax.plot(avg_by_step.index, avg_by_step.values,
            linewidth=2, color='steelblue', label='Mean RMSE')
    ax.fill_between(
        avg_by_step.index,
        avg_by_step.values - std_by_step.values,
        avg_by_step.values + std_by_step.values,
        alpha=0.2, color='steelblue', label='±1 std'
    )
    ax.set_title(f'Horizon = {horizon}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Prediction Step')
    if idx == 0:
        ax.set_ylabel('RMSE (avg. across 21 patients)')
    ax.legend()
    ax.grid(alpha=0.3)

fig.suptitle('Per-Step RMSE Degradation — Multi-Output TCN',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/per_step_rmse_degradation.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.6 — Per-Step R² Degradation (all horizons overlaid)

In [ ]:
# ── 7.6  Per-step R² degradation (all horizons overlaid) ─────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('tab10', len(multi_step_horizons))

for idx, horizon in enumerate(multi_step_horizons):
    step_dfs = [per_step_results[(rid, horizon)]
                for rid in PATIENTS
                if (rid, horizon) in per_step_results]
    if not step_dfs:
        continue
    combined = pd.concat(step_dfs)
    avg_r2 = combined.groupby('Step')['R2'].mean()
    ax.plot(avg_r2.index, avg_r2.values,
            linewidth=2, color=colors[idx], marker='o', markersize=3,
            label=f'H={horizon}')

ax.set_title('Per-Step R² Degradation by Horizon', fontsize=14, fontweight='bold')
ax.set_xlabel('Prediction Step')
ax.set_ylabel('R² (avg. across 21 patients)')
ax.legend(title='Horizon')
ax.grid(alpha=0.3)
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='R²=0.5 threshold')
plt.tight_layout()
plt.savefig('plots/per_step_r2_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.7 — Actual vs Predicted Plots (Representative Patients)

In [ ]:
# ── 7.7  Actual vs Predicted ──────────────────────────────────────────────────
VIZ_PATIENTS = ['100', '103', '109', '117']
VIZ_HORIZONS = [1, 10, 20]

for rid in VIZ_PATIENTS:
    for horizon in VIZ_HORIZONS:
        key = (rid, horizon)
        if key not in saved_preds:
            continue
        y_te, preds = saved_preds[key]
        n_show = min(500, len(y_te))

        if horizon == 1:
            # Single-step: one plot
            fig, ax = plt.subplots(figsize=(14, 4))
            ax.plot(y_te[:n_show].flatten(), label='Actual',
                    linewidth=0.8, color='steelblue')
            ax.plot(preds[:n_show].flatten(), label='Predicted',
                    linewidth=0.8, color='orangered', alpha=0.8)
            ax.set_title(f'Patient {rid} — H=1 (next sample, {1/FS*1000:.1f} ms)',
                         fontsize=12, fontweight='bold')
            ax.legend(); ax.set_ylabel('Amplitude (norm.)')
            ax.set_xlabel('Sample index'); ax.grid(alpha=0.3)
        else:
            # Multi-step: step 1 and last step
            fig, axes = plt.subplots(2, 1, figsize=(14, 8))
            axes[0].plot(y_te[:n_show, 0], label='Actual',
                         linewidth=0.8, color='steelblue')
            axes[0].plot(preds[:n_show, 0], label='Predicted',
                         linewidth=0.8, color='orangered', alpha=0.8)
            axes[0].set_title(f'Step 1 ({1/FS*1000:.1f} ms ahead)')
            axes[0].legend(); axes[0].set_ylabel('Amplitude (norm.)'); axes[0].grid(alpha=0.3)

            last = horizon - 1
            axes[1].plot(y_te[:n_show, last], label='Actual',
                         linewidth=0.8, color='steelblue')
            axes[1].plot(preds[:n_show, last], label='Predicted',
                         linewidth=0.8, color='orangered', alpha=0.8)
            axes[1].set_title(f'Step {horizon} ({horizon/FS*1000:.1f} ms ahead)')
            axes[1].legend(); axes[1].set_ylabel('Amplitude (norm.)')
            axes[1].set_xlabel('Sample index'); axes[1].grid(alpha=0.3)

            fig.suptitle(f'Patient {rid} — Multi-Output H={horizon}',
                         fontsize=13, fontweight='bold')

        plt.tight_layout()
        fname = f'plots/pred_{rid}_MO_H{horizon}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'  Saved: {fname}')

### 7.8 — Training Loss Curves (sample patients)

In [ ]:
# ── 7.8  Training loss curves ─────────────────────────────────────────────────
LOSS_VIZ_PATIENTS = ['100', '103', '117']

fig, axes = plt.subplots(len(LOSS_VIZ_PATIENTS), len(HORIZONS),
                         figsize=(5 * len(HORIZONS), 4 * len(LOSS_VIZ_PATIENTS)),
                         sharex=False)

for row, rid in enumerate(LOSS_VIZ_PATIENTS):
    for col, horizon in enumerate(HORIZONS):
        ax = axes[row, col] if len(LOSS_VIZ_PATIENTS) > 1 else axes[col]
        key = (rid, horizon)
        if key not in training_histories:
            continue
        hist = training_histories[key]
        epochs_ran = range(1, len(hist['loss']) + 1)
        ax.plot(epochs_ran, hist['loss'], label='Train', linewidth=1.2)
        ax.plot(epochs_ran, hist['val_loss'], label='Val', linewidth=1.2)
        ax.set_title(f'Patient {rid}, H={horizon}', fontsize=10)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('MSE Loss')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

fig.suptitle('Training & Validation Loss Curves',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/training_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.9 — Error Distribution Histograms

In [ ]:
# ── 7.9  Error distribution histograms ────────────────────────────────────────
ERR_VIZ_PATIENTS = ['100', '109']

fig, axes = plt.subplots(len(ERR_VIZ_PATIENTS), len(HORIZONS),
                         figsize=(5 * len(HORIZONS), 4 * len(ERR_VIZ_PATIENTS)))

for row, rid in enumerate(ERR_VIZ_PATIENTS):
    for col, horizon in enumerate(HORIZONS):
        ax = axes[row, col] if len(ERR_VIZ_PATIENTS) > 1 else axes[col]
        key = (rid, horizon)
        if key not in saved_preds:
            continue
        y_te, preds = saved_preds[key]
        errors = (y_te - preds).flatten()
        ax.hist(errors, bins=80, color='steelblue', alpha=0.7, edgecolor='white')
        ax.axvline(x=0, color='red', linestyle='--', linewidth=1)
        ax.set_title(f'Patient {rid}, H={horizon}', fontsize=10)
        ax.set_xlabel('Prediction Error')
        ax.set_ylabel('Count')
        # Annotate with mean and std
        ax.text(0.02, 0.95, f'μ={errors.mean():.4f}\nσ={errors.std():.4f}',
                transform=ax.transAxes, fontsize=8, va='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle('Prediction Error Distribution',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/error_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.10 — Patient Ranking: Best & Worst R² per Horizon

In [ ]:
# ── 7.10  Patient ranking bar charts ──────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, horizon in zip(axes.flatten(), HORIZONS):
    df_h = df_results[df_results['Horizon'] == horizon].sort_values('R2', ascending=True)
    colors = ['#e74c3c' if r2 < 0.5 else '#f39c12' if r2 < 0.8 else '#2ecc71'
              for r2 in df_h['R2']]
    ax.barh(df_h['Patient'], df_h['R2'], color=colors, edgecolor='black', linewidth=0.3)
    ax.set_title(f'Horizon = {horizon} steps ({horizon/FS*1000:.1f} ms)',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('R²')
    ax.set_ylabel('Patient ID')
    ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5)
    ax.axvline(x=0.8, color='orange', linestyle='--', alpha=0.5)
    ax.set_xlim(left=min(0, df_h['R2'].min() - 0.05))

fig.suptitle('Patient R² Rankings — Multi-Output TCN\n'
             '(🟢 ≥ 0.8  🟡 ≥ 0.5  🔴 < 0.5)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/patient_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.11 — Scatter Plot: RMSE vs R² (coloured by horizon)

In [ ]:
# ── 7.11  Scatter: RMSE vs R² ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))

for horizon in HORIZONS:
    df_h = df_results[df_results['Horizon'] == horizon]
    ax.scatter(df_h['RMSE'], df_h['R2'], s=60, alpha=0.7,
               label=f'H={horizon}', edgecolors='black', linewidth=0.3)

ax.set_xlabel('RMSE', fontsize=12)
ax.set_ylabel('R²', fontsize=12)
ax.set_title('RMSE vs R² — All Patients × Horizons', fontsize=14, fontweight='bold')
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.4, label='R²=0.5')
ax.legend(title='Horizon', fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('plots/scatter_rmse_vs_r2.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.12 — Violin Plots: R² Distribution per Horizon

In [ ]:
# ── 7.12  Violin plots ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

sns.violinplot(
    data=df_results, x='Horizon', y='R2', ax=ax,
    palette='muted', inner='box', cut=0
)
ax.set_title('R² Distribution per Horizon (Violin Plot)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Horizon (steps)')
ax.set_ylabel('R²')
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.4)
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('plots/violin_r2.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.13 — Summary Radar Chart: Mean Metrics per Horizon

In [ ]:
# ── 7.13  Radar chart ─────────────────────────────────────────────────────────
from math import pi

categories = ['R²', '1-RMSE', '1-MAE']  # normalised so higher = better
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # close polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors = sns.color_palette('tab10', len(HORIZONS))

for idx, horizon in enumerate(HORIZONS):
    df_h = df_results[df_results['Horizon'] == horizon]
    r2_mean   = df_h['R2'].mean()
    rmse_mean = df_h['RMSE'].mean()
    mae_mean  = df_h['MAE'].mean()
    values = [max(0, r2_mean), max(0, 1 - rmse_mean), max(0, 1 - mae_mean)]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=f'H={horizon}', color=colors[idx])
    ax.fill(angles, values, alpha=0.1, color=colors[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 1)
ax.set_title('Multi-Output TCN — Performance Radar\n(higher = better)',
             fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), title='Horizon')
plt.tight_layout()
plt.savefig('plots/radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Final Summary

In [ ]:
print('\n' + '=' * 80)
print('EXPERIMENT COMPLETE')
print('=' * 80)
print(f'Strategy         : Multi-Output (MIMO)')
print(f'Patients         : {len(PATIENTS)} (all MIT-BIH excl. 111 & 118)')
print(f'Horizons         : {HORIZONS}')
print(f'Total experiments: {len(df_results)}')
print(f'Epochs per model : {EPOCHS}')
print(f'Results CSV      : tcn_multioutput_results.csv')
print(f'Plots directory  : plots/')
print()
print('Horizon interpretations (at 360 Hz sampling):')
for h in HORIZONS:
    ms = h / FS * 1000
    print(f'  H={h:2d} → {ms:6.1f} ms ≈ {h/FS:.4f} s into the future')
print()
print('Best horizon (mean R²):')
best = df_results.groupby('Horizon')['R2'].mean().sort_values(ascending=False)
for h, r2 in best.items():
    print(f'  H={h:2d} : R² = {r2:.4f}')
print()
print('Plots generated:')
for f in sorted(os.listdir('plots')):
    if f.endswith('.png'):
        print(f'  📊 plots/{f}')